<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/GNN_Transformer2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## connect to /content/drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Install torch-geometric

In [3]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.6 MB/s eta 0:00:00


# GNN/Transformer based emulator with PyTorch Geometric

In [7]:
# ============================================================
# train_klein_pyg_graph_tendency_v15.py
# ------------------------------------------------------------
# PyTorch Geometric version of the Klein-topology graph tendency
# emulator for 1-layer shallow-water equations on the Klein beta-plane.
#
# This v7 script is based on the v6 emulator/evaluator line of development:
#   - fixed Klein graph
#   - multi-radius connectivity r = 1, 2, 4
#   - normalized graph inputs
#   - properly scaled physical tendencies
#   - explicit residual update scaling RES_SCALE
#   - no full-resolution Transformer in this first PyG diagnostic version
#   - same rollout, collocation, continuity/momentum/geostrophic losses
#   - same spectral UV/vorticity/KE/high-k/divergence losses
#   - checkpoint/autoresume
#
# Colab setup:
#   !pip install torch-geometric
#
# Output root:
#   /content/drive/MyDrive/AI_GRAPH_TRANS2
# ============================================================

import os, sys, glob, csv, time, random, re, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# PyTorch Geometric
# ------------------------------------------------------------
try:
    import torch_geometric
    from torch_geometric.nn import MessagePassing
    from torch_geometric.utils import degree
    print("torch_geometric version:", torch_geometric.__version__)
except Exception as e:
    raise RuntimeError(
        "PyTorch Geometric is required for this script. In Colab run:\n"
        "    !pip install torch-geometric\n"
        "then restart/rerun the notebook cell.\n"
        f"Original import error: {repr(e)}"
    )

# ------------------------------------------------------------
# Optional project import: adaptive collocation bank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
try:
    from colloc_bank import CollocBank
except Exception as e:
    CollocBank = None
    print("[Warning] Could not import CollocBank yet:", repr(e))

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# CONFIG
# ============================================================
class CFG:
    ROOT_CODE = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT  = "/content/drive/MyDrive/AI_GRAPH_TRANS2"

    DATA_DIR_CANDIDATES = [
        "/content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_adaptive_combined",
        "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_adaptive_combined",
        "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined",
    ]

    COLLOC_DIR_CANDIDATES = [
        "/content/AI_EMUL_LOCAL/klein_ml_1L_big50_adaptive_combined/colloc_raw",
        "/content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_adaptive_combined/colloc_raw",
    ]
    EXP_NAME = "klein_pyg_graph_mr1248_mp6_trans_adapt_v17"


    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 50.0 * 30.0
    FMIN = 2.0e-5

    # Normalization and residual dynamics
    ETA_SCALE = 800.0
    U_SCALE = 50.0
    KE_SCALE = 0.5 * U_SCALE**2

    # The network predicts nondimensional tendencies. These convert back
    # to physical tendency units, then RES_SCALE controls the macro-step update.
    TENDENCY_SCALE = 0.1
    RES_SCALE = 0.12

    # Graph model
    IN_CH = 3
    NODE_CH = 96
    HIDDEN = 64
    MP_BLOCKS = 6
    EDGE_RADII = (1, 2, 4, 8)
    EDGE_ATTR_DIM = 4
    DROPOUT = 0.0

    # Transformer intentionally OFF in this first PyG diagnostic.
    # Later v8 should add a coarse Klein graph Transformer, not full grid attention.
    USE_GLOBAL_TRANS = True
    GLOBAL_TRANS_HEADS = 4
    GLOBAL_TRANS_BLOCKS = 2
    GLOBAL_TOKEN_H = 16
    GLOBAL_TOKEN_W = 32

    # Training
    EPOCHS = 20
    BATCH_SIZE = 1
    LR = 5e-5
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0
    ROLL_STEPS = 2
    ROLL_GAMMA = 0.95
    MAX_WINDOWS_PER_IC = 16
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # Collocation
    N_TIME_SLICES = 8
    PTS_PER_TIME = 24

    # Loss weights
    LAMBDA_DATA = 1.5
    LAMBDA_COLL_STATE = 0.15
    LAMBDA_COLL_TEND = 0.4
    LAMBDA_CONT = 3.0
    LAMBDA_MOM = 5.0
    LAMBDA_GEO = 0.02

    LAMBDA_YCENTER = 1e-4

    LAMBDA_VMEAN = 2e-3
    LAMBDA_ETA_SMOOTH = 2e-6
    LAMBDA_KE_RATIO = 1e-3

    LAMBDA_HEMI_ETA = 5e-4


    LAMBDA_UV_MAG = 1e-6
    LAMBDA_KE = 2.0e-4

    LAMBDA_SPEC_UV = 3.0e-2
    LAMBDA_SPEC_ZETA = 1.5e-2
    LAMBDA_SPEC_KE = 5.0e-3
    LAMBDA_HIGHK = 2.0e-3
    LAMBDA_SPEC_DIV = 2.0
    LAMBDA_ETA_TEND_GRID = 1.0e-1
    LAMBDA_LOCAL_ENERGY_TEND = 0.0

    # Optional new stabilizer for graph tendency smoothness.
    # Keep weak for first PyG test.
    LAMBDA_TEND_SMOOTH = 5.0e-4

    HIGHK_FRAC = 0.55

    USE_STATE_CLAMP = True
    ETA_CLAMP = 2000.0
    UV_CLAMP = 150.0

    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 25
    AUTO_RESUME = True
    RESUME_FROM = None
    MAX_BATCHES_PER_EPOCH = None

cfg = CFG()
cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ============================================================
# BASIC HELPERS
# ============================================================
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic, n_pairs = 0, 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += len(files) - 1
    return n_ic, n_pairs


def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir, best_pairs = None, -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_dir, best_pairs = d, n_pairs
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found. Check DATA_DIR_CANDIDATES.")
    print(f"[AutoDetect] using data: {best_dir}")
    return best_dir


def autodetect_existing_dir(candidates, label="directory"):
    print(f"[AutoDetect] checking candidate {label}s...")
    for d in candidates:
        print(f"  {d} -> exists={os.path.exists(d)}")
        if os.path.exists(d):
            print(f"[AutoDetect] using {label}: {d}")
            return d
    raise RuntimeError(f"No valid {label} found from candidates.")


_step_re = re.compile(r"klein_step_(\d+)\.npz")


def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))


def clamp_state(x):
    if not cfg.USE_STATE_CLAMP:
        return x
    eta = torch.clamp(x[:, 0:1], -cfg.ETA_CLAMP, cfg.ETA_CLAMP)
    u = torch.clamp(x[:, 1:2], -cfg.UV_CLAMP, cfg.UV_CLAMP)
    v = torch.clamp(x[:, 2:3], -cfg.UV_CLAMP, cfg.UV_CLAMP)
    return torch.cat([eta, u, v], dim=1)


def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir, colloc_dir):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss_history": loss_history,
        "data_dir": data_dir,
        "colloc_dir": colloc_dir,
        "exp_name": cfg.EXP_NAME,
        "model_class": "KleinPyGGraphTendencyModelV7",
        "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
    }, path)


def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt


def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom", "train_geo",
            "train_uv_mag", "train_ke",
            "train_spec_uv", "train_spec_zeta", "train_spec_ke", "train_highk",
            "train_spec_div", "train_eta_tend_grid", "train_tend_smooth",
        ])
        for row in loss_history:
            w.writerow(row)

# ============================================================
# DATASET
# ============================================================
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None, include_keys=None):
        self.samples = []
        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        if include_keys is not None:
            include_keys = set(include_keys)
            ic_dirs = [d for d in ic_dirs if os.path.basename(d) in include_keys]
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")), key=extract_step_from_path)
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")
            if len(files) < roll_steps + 1:
                continue
            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))
            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]
            self.samples.extend(windows)
        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")
        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]
        seq, times, steps = [], [], []
        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))
        return {
            "seq": torch.from_numpy(np.stack(seq, axis=0)),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ============================================================
# KLEIN GRAPH GENERATOR
# ============================================================
def klein_wrap_index(i, j, nx, ny):
    jj = j
    ii = i
    flips = 0
    while jj < 0:
        jj += ny
        flips += 1
    while jj >= ny:
        jj -= ny
        flips += 1
    ii = ii % nx
    if flips % 2 == 1:
        ii = (nx - 1 - ii) % nx
    return ii, jj


def build_klein_multiradius_edges(nx, ny, radii=(1, 2, 4), include_self=False):
    src_list, dst_list, typ_list, attr_list = [], [], [], []
    max_r = max(radii)

    for j in range(ny):
        for i in range(nx):
            src = j * nx + i
            for kr, r in enumerate(radii):
                candidates = [
                    (i + r, j, 4 * kr + 0, +r, 0),
                    (i - r, j, 4 * kr + 1, -r, 0),
                    (i, j + r, 4 * kr + 2, 0, +r),
                    (i, j - r, 4 * kr + 3, 0, -r),
                ]
                for ii_raw, jj_raw, et, dx_raw, dy_raw in candidates:
                    ii, jj = klein_wrap_index(ii_raw, jj_raw, nx, ny)
                    dst = jj * nx + ii
                    seam = 1.0 if (jj_raw < 0 or jj_raw >= ny) else 0.0
                    src_list.append(src)
                    dst_list.append(dst)
                    typ_list.append(et)
                    attr_list.append([dx_raw / max_r, dy_raw / max_r, r / max_r, seam])

            if include_self:
                et = 4 * len(radii)
                src_list.append(src)
                dst_list.append(src)
                typ_list.append(et)
                attr_list.append([0.0, 0.0, 0.0, 0.0])

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_type = torch.tensor(typ_list, dtype=torch.long)
    edge_attr = torch.tensor(attr_list, dtype=torch.float32)
    return edge_index, edge_type, edge_attr

# ============================================================
# PyG MESSAGE PASSING MODEL
# ============================================================
class KleinEdgeMessagePassing(MessagePassing):
    def __init__(self, ch, n_edge_types, edge_attr_dim=4, hidden=64, dropout=0.0):
        super().__init__(aggr="mean", node_dim=0)
        self.edge_emb = nn.Embedding(n_edge_types, ch)
        self.msg_mlp = nn.Sequential(
            nn.Linear(3 * ch + edge_attr_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, ch),
        )
        self.node_mlp = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Linear(ch, 4 * ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * ch, ch),
        )
        self.norm_msg = nn.LayerNorm(ch)
        self.norm_out = nn.LayerNorm(ch)

    def forward(self, x, edge_index, edge_type, edge_attr):
        # x: [N, C]
        agg = self.propagate(
            edge_index=edge_index,
            x=x,
            edge_type=edge_type,
            edge_attr=edge_attr,
        )
        x = x + self.norm_msg(agg)
        x = x + self.node_mlp(self.norm_out(x))
        return x

    def message(self, x_i, x_j, edge_type, edge_attr):
        # x_j: source node feature, x_i: target node feature
        e = self.edge_emb(edge_type)
        return self.msg_mlp(torch.cat([x_j, x_i, e, edge_attr], dim=-1))


class CoarseGridTransformer(nn.Module):
    """Kept for compatibility. Disabled in v7 by default."""
    def __init__(self, ch, heads=4, blocks=2, token_h=16, token_w=32, dropout=0.0):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w
        self.pos = nn.Parameter(torch.zeros(1, token_h * token_w, ch))
        layer = nn.TransformerEncoderLayer(
            d_model=ch,
            nhead=heads,
            dim_feedforward=4 * ch,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=blocks)
        self.proj = nn.Sequential(nn.Conv2d(ch, ch, 1), nn.GELU(), nn.Conv2d(ch, ch, 1))
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)

    def forward(self, feat_grid):
        B, C, H, W = feat_grid.shape
        tok = F.adaptive_avg_pool2d(feat_grid, (self.token_h, self.token_w))
        tok = tok.flatten(2).transpose(1, 2)
        tok = self.encoder(tok + self.pos)
        y = tok.transpose(1, 2).reshape(B, C, self.token_h, self.token_w)
        y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        return feat_grid + self.proj(y)


class KleinPyGGraphTendencyModelV7(nn.Module):
    def __init__(
        self,
        nx,
        ny,
        in_ch=3,
        node_ch=96,
        hidden=64,
        mp_blocks=4,
        radii=(1, 2, 4),
        use_global_trans=False,
        global_heads=4,
        global_blocks=2,
        global_token_h=16,
        global_token_w=32,
        dropout=0.0,
    ):
        super().__init__()
        self.nx = nx
        self.ny = ny
        self.n_nodes = nx * ny
        self.radii = tuple(radii)
        self.n_edge_types = 4 * len(self.radii)

        edge_index, edge_type, edge_attr = build_klein_multiradius_edges(nx, ny, self.radii, include_self=False)
        self.register_buffer("edge_index", edge_index, persistent=False)
        self.register_buffer("edge_type", edge_type, persistent=False)
        self.register_buffer("edge_attr", edge_attr, persistent=False)

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, ny),
            torch.linspace(-1.0, 1.0, nx),
            indexing="ij",
        )
        coord = torch.stack([xx, yy], dim=-1).reshape(ny * nx, 2)
        self.register_buffer("coord", coord, persistent=False)

        self.input_proj = nn.Sequential(
            nn.Linear(in_ch + 2, node_ch),
            nn.GELU(),
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList([
            KleinEdgeMessagePassing(
                ch=node_ch,
                n_edge_types=self.n_edge_types,
                edge_attr_dim=cfg.EDGE_ATTR_DIM,
                hidden=hidden,
                dropout=dropout,
            )
            for _ in range(mp_blocks)
        ])

        self.global_trans = CoarseGridTransformer(
            node_ch,
            heads=global_heads,
            blocks=global_blocks,
            token_h=global_token_h,
            token_w=global_token_w,
            dropout=dropout,
        ) if use_global_trans else nn.Identity()

        self.feat_proj = nn.Sequential(
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
        )

        self.tend_head = nn.Sequential(
            nn.LayerNorm(node_ch),
            nn.Linear(node_ch, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(node_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        nn.init.zeros_(self.tend_head[-1].weight)
        nn.init.zeros_(self.tend_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def _grid_to_nodes(self, x):
        return x.permute(0, 2, 3, 1).reshape(x.shape[0], self.n_nodes, x.shape[1])

    def _nodes_to_grid(self, h):
        return h.reshape(h.shape[0], self.ny, self.nx, h.shape[-1]).permute(0, 3, 1, 2).contiguous()

    def _normalize_nodes(self, nodes):
        eta = nodes[..., 0:1] / cfg.ETA_SCALE
        u = nodes[..., 1:2] / cfg.U_SCALE
        v = nodes[..., 2:3] / cfg.U_SCALE
        return torch.cat([eta, u, v], dim=-1)

    def encode(self, x):
        # This v7 diagnostic keeps BATCH_SIZE=1. We loop over B for safety.
        B = x.shape[0]
        nodes_b = self._grid_to_nodes(x)
        nodes_b = self._normalize_nodes(nodes_b)

        feats = []
        for b in range(B):
            nodes = nodes_b[b]
            h = self.input_proj(torch.cat([nodes, self.coord], dim=-1))
            for blk in self.blocks:
                h = blk(h, self.edge_index, self.edge_type, self.edge_attr)
            feats.append(h)

        h_b = torch.stack(feats, dim=0)

        if not isinstance(self.global_trans, nn.Identity):
            hg = self._nodes_to_grid(h_b)
            hg = self.global_trans(hg)
            h_b = self._grid_to_nodes(hg)

        h_b = self.feat_proj(h_b)
        return h_b

    def grid_tendency(self, feat_nodes):
        xdot_nodes = self.tend_head(feat_nodes)

        xdot_nodes[..., 0:1] = xdot_nodes[..., 0:1] * cfg.TENDENCY_SCALE * cfg.ETA_SCALE
        xdot_nodes[..., 1:2] = xdot_nodes[..., 1:2] * cfg.TENDENCY_SCALE * cfg.U_SCALE
        xdot_nodes[..., 2:3] = xdot_nodes[..., 2:3] * cfg.TENDENCY_SCALE * cfg.U_SCALE

        return self._nodes_to_grid(xdot_nodes)

    def forward_one_step(self, x, dt_macro):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return clamp_state(x + cfg.RES_SCALE * dt_macro * xdot)

    def _sample_local_grid(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(field_1b, grid, mode="bilinear", padding_mode="border", align_corners=True)
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_nodes_1b, x_norm, y_norm, tau):
        feat_grid_1b = self._nodes_to_grid(feat_nodes_1b)
        feat_local = self._sample_local_grid(feat_grid_1b, x_norm.detach(), y_norm.detach())
        state0_local = self._sample_local_grid(x_1b, x_norm.detach(), y_norm.detach())
        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        delta = self.query_mlp(torch.cat([feat_local, state0_local, q], dim=-1))
        return state0_local + tau.unsqueeze(-1) * delta

# ============================================================
# COLLOCATION AND LOSSES
# ============================================================
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    """
    Adaptive FD-activity collocation sampler.

    Half of the points are sampled uniformly in each time bin.
    Half are sampled with probability proportional to FD tendency activity.

    This focuses physics enforcement near dynamically active wave fronts
    while preserving unbiased background coverage.
    """
    n_request = max(n_time_slices * pts_per_time * 12, 256)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    deta = np.asarray(base["deta_dt_fd"]).reshape(-1)
    du   = np.asarray(base["duc_dt_fd"]).reshape(-1)
    dv   = np.asarray(base["dvc_dt_fd"]).reshape(-1)

    activity = np.abs(deta) + 0.2 * np.abs(du) + 0.2 * np.abs(dv)
    activity = activity + 1.0e-12

    chosen = []

    for b in bins:
        if len(b) == 0:
            continue

        take = min(pts_per_time, len(b))
        n_uniform = take // 2
        n_adapt = take - n_uniform

        # Uniform half
        if n_uniform > 0:
            chosen_uniform = np.random.choice(b, size=n_uniform, replace=False)
        else:
            chosen_uniform = np.array([], dtype=int)

        # Activity-weighted half
        if n_adapt > 0:
            weights = activity[b].astype(np.float64)
            weights = weights / (weights.sum() + 1.0e-12)
            chosen_adapt = np.random.choice(b, size=n_adapt, replace=True, p=weights)
        else:
            chosen_adapt = np.array([], dtype=int)

        chosen.append(np.concatenate([chosen_uniform, chosen_adapt]))

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        out[k] = arr[chosen] if arr.ndim >= 1 and arr.shape[0] == len(t_sec) else arr

    return out


def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)


def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)
    tau = torch.clamp((t_sec - t_start) / dt_macro, 0.0, 1.0)
    x_norm = ((2.0 * x_m / cfg.Lx) - 1.0).clone().detach().requires_grad_(True)
    y_norm = ((2.0 * y_m / cfg.Ly) - 1.0).clone().detach().requires_grad_(True)
    tau = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat, u_hat, v_hat = state_hat[:, 0], state_hat[:, 1], state_hat[:, 2]
    h_hat = cfg.H + eta_hat

    def grads_of(field):
        return torch.autograd.grad(field.sum(), [x_norm, y_norm, tau], create_graph=True, retain_graph=True)

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn, u_yn, u_tau = grads_of(u_hat)
    v_xn, v_yn, v_tau = grads_of(v_hat)
    eta_x, eta_y = eta_xn * (2.0 / cfg.Lx), eta_yn * (2.0 / cfg.Ly)
    u_x, u_y = u_xn * (2.0 / cfg.Lx), u_yn * (2.0 / cfg.Ly)
    v_x, v_y = v_xn * (2.0 / cfg.Lx), v_yn * (2.0 / cfg.Ly)
    eta_t, u_t, v_t = eta_tau / dt_macro, u_tau / dt_macro, v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x, hv_y = hu_xn * (2.0 / cfg.Lx), hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true = torch.as_tensor(colloc["uc"], dtype=torch.float32, device=device)
    v_true = torch.as_tensor(colloc["vc"], dtype=torch.float32, device=device)
    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd = torch.as_tensor(colloc["duc_dt_fd"], dtype=torch.float32, device=device)
    dvc_dt_fd = torch.as_tensor(colloc["dvc_dt_fd"], dtype=torch.float32, device=device)
    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    loss_coll_state = ((eta_hat - eta_true) ** 2).mean() + ((u_hat - u_true) ** 2).mean() + ((v_hat - v_true) ** 2).mean()
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean() + ((u_t - duc_dt_fd) ** 2).mean() + ((v_t - dvc_dt_fd) ** 2).mean()
    loss_cont = (eta_t + hu_x + hv_y).pow(2).mean()

    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * eta_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * eta_y
    loss_mom = ((resid_u / (1.0 + torch.abs(u_hat))) ** 2).mean() + ((resid_v / (1.0 + torch.abs(v_hat))) ** 2).mean()

    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g = (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    u_nd, v_nd = u_hat / cfg.U_SCALE, v_hat / cfg.U_SCALE
    loss_uv_mag = (u_nd ** 2 + v_nd ** 2).mean()

    ke_hat = 0.5 * (u_hat ** 2 + v_hat ** 2)
    ke_true = 0.5 * (u_true ** 2 + v_true ** 2)
    loss_ke = (((ke_hat - ke_true) / cfg.KE_SCALE) ** 2).mean()

    # --------------------------------------------------------
    # Local kinetic/potential energy tendency consistency
    # --------------------------------------------------------
    # KE tendency:
    #   d/dt(0.5*(u^2+v^2)) = u*u_t + v*v_t
    #
    # Linearized shallow-water PE tendency:
    #   d/dt(0.5*g*eta^2) = g*eta*eta_t
    #
    # This penalizes incorrect local exchange between kinetic and
    # potential energy reservoirs. It is weakly related to the
    # pressure-work / omega-alpha conversion mechanism.
    # --------------------------------------------------------
    ke_t_hat = u_hat * u_t + v_hat * v_t
    pe_t_hat = cfg.G * eta_hat * eta_t
    e_t_hat = ke_t_hat + pe_t_hat

    ke_t_fd = u_true * duc_dt_fd + v_true * dvc_dt_fd
    pe_t_fd = cfg.G * eta_true * deta_dt_fd
    e_t_fd = ke_t_fd + pe_t_fd

    e_t_scale = e_t_fd.pow(2).mean().detach().sqrt() + 1e-8
    loss_local_energy_tend = (((e_t_hat - e_t_fd) / e_t_scale) ** 2).mean()

    return (
        loss_coll_state,
        loss_coll_tend,
        loss_cont,
        loss_mom,
        loss_geo,
        loss_uv_mag,
        loss_ke,
        loss_local_energy_tend,
    )



# ============================================================
# GRID OPERATORS AND SPECTRAL LOSSES
# ============================================================
def fd_grad_x(q):
    return (torch.roll(q, shifts=-1, dims=-1) - torch.roll(q, shifts=1, dims=-1)) / (2.0 * cfg.DX)


def fd_grad_y_reflect(q):
    qp = torch.empty_like(q)
    qm = torch.empty_like(q)
    qp[:, :, :-1, :] = q[:, :, 1:, :]
    qp[:, :, -1, :] = q[:, :, -1, :]
    qm[:, :, 1:, :] = q[:, :, :-1, :]
    qm[:, :, 0, :] = q[:, :, 0, :]
    return (qp - qm) / (2.0 * cfg.DY)


def divergence_grid(x):
    return fd_grad_x(x[:, 1:2]) + fd_grad_y_reflect(x[:, 2:3])


def vorticity_grid(x):
    return fd_grad_x(x[:, 2:3]) - fd_grad_y_reflect(x[:, 1:2])


def laplacian_grid(q):
    q_xp = torch.roll(q, shifts=-1, dims=-1)
    q_xm = torch.roll(q, shifts=1, dims=-1)
    q_yp = torch.empty_like(q)
    q_ym = torch.empty_like(q)
    q_yp[:, :, :-1, :] = q[:, :, 1:, :]
    q_yp[:, :, -1, :] = q[:, :, -1, :]
    q_ym[:, :, 1:, :] = q[:, :, :-1, :]
    q_ym[:, :, 0, :] = q[:, :, 0, :]
    return (q_xp - 2.0 * q + q_xm) / cfg.DX**2 + (q_yp - 2.0 * q + q_ym) / cfg.DY**2


def tendency_smooth_loss(xdot_grid):
    # Weak tendency Laplacian penalty, normalized per component.
    eta_lap = laplacian_grid(xdot_grid[:, 0:1])
    u_lap = laplacian_grid(xdot_grid[:, 1:2])
    v_lap = laplacian_grid(xdot_grid[:, 2:3])
    return (
        (eta_lap / (cfg.ETA_SCALE + 1e-8)).pow(2).mean()
        + (u_lap / (cfg.U_SCALE + 1e-8)).pow(2).mean()
        + (v_lap / (cfg.U_SCALE + 1e-8)).pow(2).mean()
    )

def y_center_penalty(pred):
    """
    Penalize energy away from equatorial waveguide.
    Helps RH/wave cases avoid filling entire domain.
    """

    v = pred[:, 2:3]

    y = torch.linspace(
        -1.0, 1.0,
        pred.shape[-2],
        device=pred.device
    ).view(1, 1, -1, 1)

    # weak equatorial Gaussian mask
    mask = 1.0 - torch.exp(-(y / 0.35) ** 2)

    return (mask * v**2).mean()

def vmean_loss(pred):
    return pred[:, 2:3].mean().pow(2)


def eta_smooth_loss(pred):
    eta = pred[:, 0:1]
    eta_lap = laplacian_grid(eta)
    return (eta_lap / (cfg.ETA_SCALE + 1e-8)).pow(2).mean()


def ke_ratio_loss(pred, truth):
    ke_pred = torch.mean(0.5 * (pred[:, 1:2]**2 + pred[:, 2:3]**2))
    ke_true = torch.mean(0.5 * (truth[:, 1:2]**2 + truth[:, 2:3]**2))
    return ((ke_pred / (ke_true + 1e-8)) - 1.0).pow(2)

def hemispheric_eta_balance_loss(pred):
    eta = pred[:, 0:1]   # [B,1,NY,NX]

    ny = eta.shape[-2]
    south = eta[:, :, :ny//2, :]
    north = eta[:, :, ny//2:, :]

    # Penalize different mean eta between hemispheres
    return (north.mean() - south.mean()).pow(2)

def spectral_masks(ny, nx, device):
    ky = torch.fft.fftfreq(ny, d=cfg.DY).to(device)[:, None]
    kx = torch.fft.rfftfreq(nx, d=cfg.DX).to(device)[None, :]
    kr = torch.sqrt(kx ** 2 + ky ** 2)
    kr_norm = kr / (kr.max() + 1e-12)
    high = (kr_norm >= cfg.HIGHK_FRAC).float()
    weight_k2 = kr_norm ** 2
    return kr_norm[None, None], high[None, None], weight_k2[None, None]


def spectral_arakawa_losses(pred, truth):
    B, C, NY, NX = pred.shape
    u_p, v_p = pred[:, 1:2], pred[:, 2:3]
    u_t, v_t = truth[:, 1:2], truth[:, 2:3]
    kr_norm, high, weight_k2 = spectral_masks(NY, NX, pred.device)

    Uh_p, Vh_p = torch.fft.rfft2(u_p, norm="ortho"), torch.fft.rfft2(v_p, norm="ortho")
    Uh_t, Vh_t = torch.fft.rfft2(u_t, norm="ortho"), torch.fft.rfft2(v_t, norm="ortho")
    KEp = Uh_p.abs() ** 2 + Vh_p.abs() ** 2
    KEt = Uh_t.abs() ** 2 + Vh_t.abs() ** 2
    denom_ke = KEt.mean().detach() + 1e-8

    loss_spec_uv = (((KEp - KEt) / denom_ke) ** 2).mean()
    loss_spec_ke = (weight_k2 * ((KEp - KEt) / denom_ke) ** 2).mean()

    Zh_p = torch.fft.rfft2(vorticity_grid(pred), norm="ortho")
    Zh_t = torch.fft.rfft2(vorticity_grid(truth), norm="ortho")
    ZEp, ZEt = Zh_p.abs() ** 2, Zh_t.abs() ** 2
    denom_z = ZEt.mean().detach() + 1e-12
    loss_spec_zeta = (((ZEp - ZEt) / denom_z) ** 2).mean()
    loss_highk = (high * KEp / denom_ke).mean() + (high * ZEp / denom_z).mean()

    Dh_p = torch.fft.rfft2(divergence_grid(pred), norm="ortho")
    Dh_t = torch.fft.rfft2(divergence_grid(truth), norm="ortho")
    DEp, DEt = Dh_p.abs() ** 2, Dh_t.abs() ** 2
    denom_d = DEt.mean().detach() + 1e-14
    loss_spec_div = (((DEp - DEt) / denom_d) ** 2).mean()

    return loss_spec_uv, loss_spec_zeta, loss_spec_ke, loss_highk, loss_spec_div

def vmean_loss(pred):
    return pred[:, 2:3].mean().pow(2)


def eta_smooth_loss(pred):
    eta = pred[:, 0:1]
    eta_lap = laplacian_grid(eta)
    return (eta_lap / (cfg.ETA_SCALE + 1e-8)).pow(2).mean()


def ke_ratio_loss(pred, truth):
    ke_pred = torch.mean(0.5 * (pred[:, 1:2]**2 + pred[:, 2:3]**2))
    ke_true = torch.mean(0.5 * (truth[:, 1:2]**2 + truth[:, 2:3]**2))
    return ((ke_pred / (ke_true + 1e-8)) - 1.0).pow(2)

def eta_tendency_grid_loss(xdot_grid, x, truth_next):
    eta_t_fd = (truth_next[:, 0:1] - x[:, 0:1]) / cfg.DT_MACRO
    eta_t_ml = xdot_grid[:, 0:1]
    scale = eta_t_fd.pow(2).mean().detach().sqrt() + 1e-8
    return (((eta_t_ml - eta_t_fd) / scale) ** 2).mean()

# ============================================================
# TRAINING
# ============================================================
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)
    colloc_dir = autodetect_existing_dir(cfg.COLLOC_DIR_CANDIDATES, label="collocation directory")
    if CollocBank is None:
        raise RuntimeError("CollocBank import failed. Check sys.path and colloc_bank.py location.")
    colloc_bank = CollocBank(colloc_dir, verbose=True)

    VAL_KEYS = {"test_rh_00", "wave_00", "wave_01", "mix_04"}

    dataset = SWWindowDataset(data_dir, roll_steps=cfg.ROLL_STEPS, max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC)
    dataset.samples = [s for s in dataset.samples if s[1] not in VAL_KEYS]
    print(f"[Dataset] training windows after hold-out = {len(dataset.samples)}")
    print("[Dataset] hold-out validation ICs:", sorted(VAL_KEYS))

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = KleinPyGGraphTendencyModelV7(
        nx=cfg.NX,
        ny=cfg.NY,
        in_ch=cfg.IN_CH,
        node_ch=cfg.NODE_CH,
        hidden=cfg.HIDDEN,
        mp_blocks=cfg.MP_BLOCKS,
        radii=cfg.EDGE_RADII,
        use_global_trans=cfg.USE_GLOBAL_TRANS,
        global_heads=cfg.GLOBAL_TRANS_HEADS,
        global_blocks=cfg.GLOBAL_TRANS_BLOCKS,
        global_token_h=cfg.GLOBAL_TOKEN_H,
        global_token_w=cfg.GLOBAL_TOKEN_W,
        dropout=cfg.DROPOUT,
    ).to(device)

    print(f"[Graph] nodes = {cfg.NX * cfg.NY:,}, edges = {model.edge_index.shape[1]:,}, radii = {cfg.EDGE_RADII}")
    print(f"[Model] trainable parameters = {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch, loss_history = 0, []
    resume_path = cfg.RESUME_FROM
    auto_last = os.path.join(cfg.CKPT_DIR, "last_model.pt")
    if resume_path is None and cfg.AUTO_RESUME and os.path.exists(auto_last):
        resume_path = auto_last
    if resume_path is not None and os.path.exists(resume_path):
        print(f"[Resume] loading checkpoint: {resume_path}")
        loaded_epoch, loaded_history, _ = load_checkpoint(resume_path, model, optimizer)
        start_epoch, loss_history = loaded_epoch + 1, loaded_history
        print(f"[Resume] continuing from epoch {start_epoch}")
    else:
        print("[Resume] no checkpoint found; starting from scratch")

    best_total = min([row[1] for row in loss_history], default=float("inf"))
    if start_epoch >= cfg.EPOCHS:
        print(f"[Train] checkpoint already reached epoch {start_epoch - 1}; EPOCHS={cfg.EPOCHS}. Nothing to do.")
        return

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] data dir              = {data_dir}")
    print(f"[Train] colloc dir            = {colloc_dir}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] ROLL_STEPS            = {cfg.ROLL_STEPS}")
    print(f"[Train] EPOCHS                = {cfg.EPOCHS}")
    print(f"[Train] RES_SCALE             = {cfg.RES_SCALE}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()
        run = {k: 0.0 for k in [
            "total", "data", "cstate", "ctend", "cont", "mom", "geo", "uv_mag", "ke",
            "spec_uv", "spec_zeta", "spec_ke", "highk", "spec_div", "eta_tend_grid", "tend_smooth",
            "ycenter", "vmean", "eta_smooth", "ke_ratio_loss"
        ]}

        n_batches_effective = 0

        for ib, batch in enumerate(loader):
            if cfg.MAX_BATCHES_PER_EPOCH is not None and ib >= cfg.MAX_BATCHES_PER_EPOCH:
                print(f"[debug] stopping epoch early at batch {ib}")
                break

            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()
            B = seq.shape[0]
            if B != 1:
                raise RuntimeError("This v7 graph version expects BATCH_SIZE=1.")

            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = loss_ctend = loss_cont = loss_mom = loss_geo = loss_uv_mag = loss_ke = 0.0
            loss_local_energy_tend = 0.0
            loss_spec_uv = loss_spec_zeta = loss_spec_ke = loss_highk = loss_spec_div = 0.0
            loss_eta_tend_grid = 0.0
            loss_tend_smooth = 0.0
            loss_ycenter = 0.0
            loss_vmean = 0.0
            loss_eta_smooth = 0.0
            loss_ke_ratio = 0.0
            loss_hemi_eta = 0.0
            n_phys = 0
            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = clamp_state(x + cfg.RES_SCALE * cfg.DT_MACRO * xdot_grid)
                truth_next = seq[:, k + 1]

                loss_data = loss_data + (cfg.ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()
                l_suv, l_szeta, l_ske, l_highk, l_sdiv = spectral_arakawa_losses(x_next, truth_next)
                loss_spec_uv += l_suv
                loss_spec_zeta += l_szeta
                loss_spec_ke += l_ske
                loss_highk += l_highk
                loss_spec_div += l_sdiv
                loss_eta_tend_grid += eta_tendency_grid_loss(xdot_grid, x, truth_next)
                loss_tend_smooth += tendency_smooth_loss(xdot_grid)
                loss_ycenter += y_center_penalty(x_next)
                loss_vmean += vmean_loss(x_next)
                loss_eta_smooth += eta_smooth_loss(x_next)
                loss_ke_ratio += ke_ratio_loss(x_next, truth_next)
                loss_hemi_eta += hemispheric_eta_balance_loss(x_next)

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])
                    if not colloc_bank.has_ic(ic_key):
                        continue
                    colloc = sample_multitime_colloc(
                        colloc_bank,
                        ic_key,
                        macro_idx,
                        cfg.N_TIME_SLICES,
                        cfg.PTS_PER_TIME,
                    )
                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_uv_mag, l_ke, l_local_energy_tend = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b + 1],
                        feat_1b=feat[b:b + 1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_uv_mag += l_uv_mag
                    loss_ke += l_ke
                    loss_local_energy_tend += l_local_energy_tend
                    n_phys += 1

                x = x_next

            loss_spec_uv /= cfg.ROLL_STEPS
            loss_spec_zeta /= cfg.ROLL_STEPS
            loss_spec_ke /= cfg.ROLL_STEPS
            loss_highk /= cfg.ROLL_STEPS
            loss_spec_div /= cfg.ROLL_STEPS
            loss_eta_tend_grid /= cfg.ROLL_STEPS
            loss_tend_smooth /= cfg.ROLL_STEPS
            loss_ycenter /= cfg.ROLL_STEPS
            loss_vmean /= cfg.ROLL_STEPS
            loss_eta_smooth /= cfg.ROLL_STEPS
            loss_ke_ratio /= cfg.ROLL_STEPS
            loss_hemi_eta /= cfg.ROLL_STEPS


            if n_phys > 0:
                loss_cstate /= n_phys
                loss_ctend /= n_phys
                loss_cont /= n_phys
                loss_mom /= n_phys
                loss_geo /= n_phys
                loss_uv_mag /= n_phys
                loss_ke /= n_phys
                loss_local_energy_tend /= n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = loss_ctend = loss_cont = loss_mom = loss_geo = loss_uv_mag = loss_ke = zero
                loss_local_energy_tend = zero

            loss = (
                cfg.LAMBDA_DATA * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND * loss_ctend
                + cfg.LAMBDA_CONT * loss_cont
                + cfg.LAMBDA_MOM * loss_mom
                + cfg.LAMBDA_GEO * loss_geo
                + cfg.LAMBDA_UV_MAG * loss_uv_mag
                + cfg.LAMBDA_KE * loss_ke
                + cfg.LAMBDA_LOCAL_ENERGY_TEND * loss_local_energy_tend
                + cfg.LAMBDA_SPEC_UV * loss_spec_uv
                + cfg.LAMBDA_SPEC_ZETA * loss_spec_zeta
                + cfg.LAMBDA_SPEC_KE * loss_spec_ke
                + cfg.LAMBDA_HIGHK * loss_highk
                + cfg.LAMBDA_SPEC_DIV * loss_spec_div
                + cfg.LAMBDA_ETA_TEND_GRID * loss_eta_tend_grid
                + cfg.LAMBDA_TEND_SMOOTH * loss_tend_smooth
                + cfg.LAMBDA_YCENTER * loss_ycenter
                + cfg.LAMBDA_VMEAN * loss_vmean
                + cfg.LAMBDA_ETA_SMOOTH * loss_eta_smooth
                + cfg.LAMBDA_KE_RATIO * loss_ke_ratio
                + cfg.LAMBDA_HEMI_ETA * loss_hemi_eta
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                for name, val in [
                    ("data", loss_data), ("cstate", loss_cstate), ("ctend", loss_ctend), ("cont", loss_cont),
                    ("mom", loss_mom), ("geo", loss_geo), ("uv_mag", loss_uv_mag), ("ke", loss_ke),
                    ("spec_uv", loss_spec_uv), ("spec_zeta", loss_spec_zeta), ("spec_ke", loss_spec_ke),
                    ("highk", loss_highk), ("spec_div", loss_spec_div),
                    ("eta_tend_grid", loss_eta_tend_grid), ("tend_smooth", loss_tend_smooth),
                ]:
                    print(f"  {name:16s}=", float(val.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            vals = {
                "total": loss,
                "data": loss_data,
                "cstate": loss_cstate,
                "ctend": loss_ctend,
                "cont": loss_cont,
                "mom": loss_mom,
                "geo": loss_geo,
                "uv_mag": loss_uv_mag,
                "ke": loss_ke,
                "spec_uv": loss_spec_uv,
                "spec_zeta": loss_spec_zeta,
                "spec_ke": loss_spec_ke,
                "highk": loss_highk,
                "spec_div": loss_spec_div,
                "eta_tend_grid": loss_eta_tend_grid,
                "tend_smooth": loss_tend_smooth,
                "ycenter": loss_ycenter,
                "vmean": loss_vmean,
                "eta_smooth": loss_eta_smooth,
                "ke_ratio_loss": loss_ke_ratio,
            }
            for kk, vv in vals.items():
                run[kk] += float(vv.detach().cpu())
            n_batches_effective += 1

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} mom={float(loss_mom.detach().cpu()):.6e} "
                    f"localE={float(loss_local_energy_tend.detach().cpu()):.6e} "
                    f"specdiv={float(loss_spec_div.detach().cpu()):.6e} etatend={float(loss_eta_tend_grid.detach().cpu()):.6e} "
                    f"smooth={float(loss_tend_smooth.detach().cpu()):.6e}"
                )

        n_batches = max(n_batches_effective, 1)
        ep = {k: v / n_batches for k, v in run.items()}
        loss_history.append([
            epoch,
            ep["total"], ep["data"], ep["cstate"], ep["ctend"], ep["cont"], ep["mom"], ep["geo"],
            ep["uv_mag"], ep["ke"], ep["spec_uv"], ep["spec_zeta"], ep["spec_ke"], ep["highk"],
            ep["spec_div"], ep["eta_tend_grid"], ep["tend_smooth"],
        ])

        print(
            f"Epoch {epoch:03d} done | total={ep['total']:.6e} | data={ep['data']:.6e} | "
            f"cstate={ep['cstate']:.6e} | ctend={ep['ctend']:.6e} | cont={ep['cont']:.6e} | mom={ep['mom']:.6e} | "
            f"geo={ep['geo']:.6e} | uv_mag={ep['uv_mag']:.6e} | ke={ep['ke']:.6e} | "
            f"specuv={ep['spec_uv']:.6e} | specz={ep['spec_zeta']:.6e} | specke={ep['spec_ke']:.6e} | "
            f"highk={ep['highk']:.6e} | specdiv={ep['spec_div']:.6e} | "
            f"etatend={ep['eta_tend_grid']:.6e} | smooth={ep['tend_smooth']:.6e} | "
            f" ycenter={ep['ycenter']:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir, colloc_dir)
        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"), model, optimizer, epoch, loss_history, data_dir, colloc_dir)
        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

        if ep["total"] < best_total:
            best_total = ep["total"]
            best_ckpt = os.path.join(cfg.CKPT_DIR, "best_model.pt")
            save_checkpoint(best_ckpt, model, optimizer, epoch, loss_history, data_dir, colloc_dir)
            print(f"[Best] epoch={epoch:03d} train_total={ep['total']:.6e} -> {best_ckpt}")

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir, colloc_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)
    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

# ============================================================
if __name__ == "__main__":
    train()


torch_geometric version: 2.7.0
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_adaptive_combined
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_adaptive_combined
     valid IC dirs = 8, total pairs = 192
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
     valid IC dirs = 68, total pairs = 1632
[AutoDetect] using data: /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
[AutoDetect] checking candidate collocation directorys...
  /content/AI_EMUL_LOCAL/klein_ml_1L_big50_adaptive_combined/colloc_raw -> exists=False
  /content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_adaptive_combined/colloc_raw -> exists=True
[AutoDetect] using collocation directory: /content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_adaptive_combined/colloc_raw
[CollocBank] loaded vort_04 | samples=23424 | macro_groups=25
[CollocBank] lo

/tmp/ipykernel_915/428165353.py:449: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=blocks)


[Graph] nodes = 32,768, edges = 524,288, radii = (1, 2, 4, 8)
[Model] trainable parameters = 945,382
[Resume] loading checkpoint: /content/drive/MyDrive/AI_GRAPH_TRANS2/training_runs/klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean/last_model.pt
[Resume] continuing from epoch 20
[Train] checkpoint already reached epoch 19; EPOCHS=20. Nothing to do.


# Evaluation of GNN/Transformer

In [9]:
# ============================================================
# eval_klein_pyg_graph_tendency_v9.py
# ------------------------------------------------------------
# Evaluation script for the PyTorch Geometric Klein graph
# tendency emulator v7.
#
# Matches:
#   train_klein_pyg_graph_tendency_v9.py
#
# Saves:
#   - summary_eval.csv
#   - all_rollout_metrics.csv
#   - per-IC rollout_metrics.csv
#   - RMSE plots
#   - KE plots
#   - divergence RMSE / spectrum plots
#   - final eta/u/v comparison plots
#   - intermediate snapshot plots
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PyTorch Geometric
# ------------------------------------------------------------
try:
    import torch_geometric
    from torch_geometric.nn import MessagePassing
    print("torch_geometric version:", torch_geometric.__version__)
except Exception as e:
    raise RuntimeError(
        "PyTorch Geometric is required. In Colab run:\n"
        "    !pip install torch-geometric\n"
        f"Original import error: {repr(e)}"
    )

# ============================================================
# CONFIG
# ============================================================
class CFG:
    ROOT_DATA_CANDIDATES = [
        "/content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_adaptive_combined",
        "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_adaptive_combined",
        "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined",
    ]

    ROOT_RUN = "/content/drive/MyDrive/AI_GRAPH_TRANS2"
    RUN_NAME = "klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean"
    RUN_DIR = os.path.join(ROOT_RUN, "training_runs", RUN_NAME)

    # For short diagnostic runs, last_model.pt is usually safest.
    # For completed longer runs, you may switch to best_model.pt.
    CKPT_NAME = "last_model.pt"
    CKPT_PATH = os.path.join(RUN_DIR, CKPT_NAME)

    OUT_DIR = os.path.join(RUN_DIR, "evaluation_v16_raw_diff")
    SNAPSHOT_DIR = os.path.join(OUT_DIR, "snapshots")

    # Defaults; overwritten from checkpoint config where available.
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    DT_MACRO = 50.0 * 30.0

    ETA_SCALE = 800.0
    U_SCALE = 50.0
    TENDENCY_SCALE = 0.1
    RES_SCALE = 0.15

    IN_CH = 3
    NODE_CH = 96
    HIDDEN = 64
    MP_BLOCKS = 6
    EDGE_RADII = (1, 2, 4, 8)
    EDGE_ATTR_DIM = 4
    DROPOUT = 0.0

    USE_GLOBAL_TRANS = True
    GLOBAL_TRANS_HEADS = 4
    GLOBAL_TRANS_BLOCKS = 2
    GLOBAL_TOKEN_H = 16
    GLOBAL_TOKEN_W = 32

    HELDOUT_KEYS = [
        "test_rh_00",
        "wave_00",
        "wave_01",
        "mix_04",
    ]

    MAX_ROLLOUT_STEPS = None
    SAVE_LEADS = list(range(1, 25))

    USE_INFERENCE_DIFFUSION = True
    NU_INFER = 5.0e4
    DIFFUSE_ETA = True
    DIFFUSE_UV = True

    # Diagnostic clamps. Keep these on for first evaluations to avoid
    # meaningless color scales if an unstable checkpoint is tested.
    USE_EVAL_CLAMP = True
    ETA_EVAL_CLAMP = 1200.0
    UV_EVAL_CLAMP = 200.0

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.SNAPSHOT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATA AUTODETECT
# ============================================================
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic, n_pairs = 0, 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += len(files) - 1
    return n_ic, n_pairs


def autodetect_data_dir():
    print("[AutoDetect] checking data roots...")
    best_dir, best_pairs = None, -1
    for d in cfg.ROOT_DATA_CANDIDATES:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_dir, best_pairs = d, n_pairs
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid DATA_DIR found.")
    print("[AutoDetect] using DATA_DIR:", best_dir)
    return best_dir

DATA_DIR = autodetect_data_dir()

# ============================================================
# KLEIN GRAPH
# ============================================================
def klein_wrap_index(i, j, nx, ny):
    jj = j
    ii = i
    flips = 0
    while jj < 0:
        jj += ny
        flips += 1
    while jj >= ny:
        jj -= ny
        flips += 1
    ii = ii % nx
    if flips % 2 == 1:
        ii = (nx - 1 - ii) % nx
    return ii, jj


def build_klein_multiradius_edges(nx, ny, radii=(1, 2, 4), include_self=False):
    src_list, dst_list, typ_list, attr_list = [], [], [], []
    max_r = max(radii)

    for j in range(ny):
        for i in range(nx):
            src = j * nx + i
            for kr, r in enumerate(radii):
                candidates = [
                    (i + r, j, 4 * kr + 0, +r, 0),
                    (i - r, j, 4 * kr + 1, -r, 0),
                    (i, j + r, 4 * kr + 2, 0, +r),
                    (i, j - r, 4 * kr + 3, 0, -r),
                ]
                for ii_raw, jj_raw, et, dx_raw, dy_raw in candidates:
                    ii, jj = klein_wrap_index(ii_raw, jj_raw, nx, ny)
                    dst = jj * nx + ii
                    seam = 1.0 if (jj_raw < 0 or jj_raw >= ny) else 0.0
                    src_list.append(src)
                    dst_list.append(dst)
                    typ_list.append(et)
                    attr_list.append([dx_raw / max_r, dy_raw / max_r, r / max_r, seam])

            if include_self:
                et = 4 * len(radii)
                src_list.append(src)
                dst_list.append(src)
                typ_list.append(et)
                attr_list.append([0.0, 0.0, 0.0, 0.0])

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_type = torch.tensor(typ_list, dtype=torch.long)
    edge_attr = torch.tensor(attr_list, dtype=torch.float32)
    return edge_index, edge_type, edge_attr

# ============================================================
# PyG MODEL
# ============================================================
class KleinEdgeMessagePassing(MessagePassing):
    def __init__(self, ch, n_edge_types, edge_attr_dim=4, hidden=64, dropout=0.0):
        super().__init__(aggr="mean", node_dim=0)
        self.edge_emb = nn.Embedding(n_edge_types, ch)
        self.msg_mlp = nn.Sequential(
            nn.Linear(3 * ch + edge_attr_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, ch),
        )
        self.node_mlp = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Linear(ch, 4 * ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * ch, ch),
        )
        self.norm_msg = nn.LayerNorm(ch)
        self.norm_out = nn.LayerNorm(ch)

    def forward(self, x, edge_index, edge_type, edge_attr):
        agg = self.propagate(
            edge_index=edge_index,
            x=x,
            edge_type=edge_type,
            edge_attr=edge_attr,
        )
        x = x + self.norm_msg(agg)
        x = x + self.node_mlp(self.norm_out(x))
        return x

    def message(self, x_i, x_j, edge_type, edge_attr):
        e = self.edge_emb(edge_type)
        return self.msg_mlp(torch.cat([x_j, x_i, e, edge_attr], dim=-1))


class CoarseGridTransformer(nn.Module):
    def __init__(self, ch, heads=4, blocks=2, token_h=16, token_w=32, dropout=0.0):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w
        self.pos = nn.Parameter(torch.zeros(1, token_h * token_w, ch))
        layer = nn.TransformerEncoderLayer(
            d_model=ch,
            nhead=heads,
            dim_feedforward=4 * ch,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=blocks)
        self.proj = nn.Sequential(nn.Conv2d(ch, ch, 1), nn.GELU(), nn.Conv2d(ch, ch, 1))
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)

    def forward(self, feat_grid):
        B, C, H, W = feat_grid.shape
        tok = F.adaptive_avg_pool2d(feat_grid, (self.token_h, self.token_w))
        tok = tok.flatten(2).transpose(1, 2)
        tok = self.encoder(tok + self.pos)
        y = tok.transpose(1, 2).reshape(B, C, self.token_h, self.token_w)
        y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        return feat_grid + self.proj(y)


class KleinPyGGraphTendencyModelV7(nn.Module):
    def __init__(
        self,
        nx,
        ny,
        in_ch=3,
        node_ch=96,
        hidden=64,
        mp_blocks=6,
        radii=(1, 2, 4, 8),
        use_global_trans = True,
        global_heads=4,
        global_blocks=2,
        global_token_h=16,
        global_token_w=32,
        dropout=0.0,
    ):
        super().__init__()
        self.nx = nx
        self.ny = ny
        self.n_nodes = nx * ny
        self.radii = tuple(radii)
        self.n_edge_types = 4 * len(self.radii)

        edge_index, edge_type, edge_attr = build_klein_multiradius_edges(nx, ny, self.radii, include_self=False)
        self.register_buffer("edge_index", edge_index, persistent=False)
        self.register_buffer("edge_type", edge_type, persistent=False)
        self.register_buffer("edge_attr", edge_attr, persistent=False)

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, ny),
            torch.linspace(-1.0, 1.0, nx),
            indexing="ij",
        )
        coord = torch.stack([xx, yy], dim=-1).reshape(ny * nx, 2)
        self.register_buffer("coord", coord, persistent=False)

        self.input_proj = nn.Sequential(
            nn.Linear(in_ch + 2, node_ch),
            nn.GELU(),
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList([
            KleinEdgeMessagePassing(
                ch=node_ch,
                n_edge_types=self.n_edge_types,
                edge_attr_dim=cfg.EDGE_ATTR_DIM,
                hidden=hidden,
                dropout=dropout,
            )
            for _ in range(mp_blocks)
        ])

        self.global_trans = CoarseGridTransformer(
            node_ch,
            heads=global_heads,
            blocks=global_blocks,
            token_h=global_token_h,
            token_w=global_token_w,
            dropout=dropout,
        ) if use_global_trans else nn.Identity()

        self.feat_proj = nn.Sequential(
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
            nn.Linear(node_ch, node_ch),
            nn.GELU(),
        )

        self.tend_head = nn.Sequential(
            nn.LayerNorm(node_ch),
            nn.Linear(node_ch, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        nn.init.zeros_(self.tend_head[-1].weight)
        nn.init.zeros_(self.tend_head[-1].bias)

    def _grid_to_nodes(self, x):
        return x.permute(0, 2, 3, 1).reshape(x.shape[0], self.n_nodes, x.shape[1])

    def _nodes_to_grid(self, h):
        return h.reshape(h.shape[0], self.ny, self.nx, h.shape[-1]).permute(0, 3, 1, 2).contiguous()

    def _normalize_nodes(self, nodes):
        eta = nodes[..., 0:1] / cfg.ETA_SCALE
        u = nodes[..., 1:2] / cfg.U_SCALE
        v = nodes[..., 2:3] / cfg.U_SCALE
        return torch.cat([eta, u, v], dim=-1)

    def encode(self, x):
        B = x.shape[0]
        nodes_b = self._grid_to_nodes(x)
        nodes_b = self._normalize_nodes(nodes_b)

        feats = []
        for b in range(B):
            nodes = nodes_b[b]
            h = self.input_proj(torch.cat([nodes, self.coord], dim=-1))
            for blk in self.blocks:
                h = blk(h, self.edge_index, self.edge_type, self.edge_attr)
            feats.append(h)

        h_b = torch.stack(feats, dim=0)

        if not isinstance(self.global_trans, nn.Identity):
            hg = self._nodes_to_grid(h_b)
            hg = self.global_trans(hg)
            h_b = self._grid_to_nodes(hg)

        h_b = self.feat_proj(h_b)
        return h_b

    def grid_tendency(self, feat_nodes):
        xdot_nodes = self.tend_head(feat_nodes)
        xdot_nodes[..., 0:1] = xdot_nodes[..., 0:1] * cfg.TENDENCY_SCALE * cfg.ETA_SCALE
        xdot_nodes[..., 1:2] = xdot_nodes[..., 1:2] * cfg.TENDENCY_SCALE * cfg.U_SCALE
        xdot_nodes[..., 2:3] = xdot_nodes[..., 2:3] * cfg.TENDENCY_SCALE * cfg.U_SCALE
        return self._nodes_to_grid(xdot_nodes)

    def forward_one_step(self, x, dt_macro):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return x + cfg.RES_SCALE * dt_macro * xdot, xdot

# ============================================================
# LOAD CHECKPOINT
# ============================================================
print("[Load] checkpoint:", cfg.CKPT_PATH)
if not os.path.exists(cfg.CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {cfg.CKPT_PATH}")

ckpt = torch.load(cfg.CKPT_PATH, map_location=device)


if "config" in ckpt and len(ckpt["config"]) > 0:
    print("[Load] updating config from checkpoint")
    for key in [
        "NX", "NY", "Lx", "Ly", "H", "DT_MACRO",
        "ETA_SCALE", "U_SCALE", "TENDENCY_SCALE", "RES_SCALE",
        "IN_CH", "NODE_CH", "HIDDEN", "MP_BLOCKS", "EDGE_RADII", "EDGE_ATTR_DIM",
        "DROPOUT", "USE_GLOBAL_TRANS", "GLOBAL_TRANS_HEADS", "GLOBAL_TRANS_BLOCKS",
        "GLOBAL_TOKEN_H", "GLOBAL_TOKEN_W",
    ]:
        if key in ckpt["config"]:
            setattr(cfg, key, ckpt["config"][key])
else:
    print("[Load] checkpoint config missing/empty; using evaluation CFG defaults")

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY

print("[Config]")
print("  RUN_NAME         =", cfg.RUN_NAME)
print("  CKPT_PATH        =", cfg.CKPT_PATH)
print("  NX, NY           =", cfg.NX, cfg.NY)
print("  NODE_CH          =", cfg.NODE_CH)
print("  HIDDEN           =", cfg.HIDDEN)
print("  MP_BLOCKS        =", cfg.MP_BLOCKS)
print("  EDGE_RADII       =", cfg.EDGE_RADII)
print("  USE_GLOBAL_TRANS =", cfg.USE_GLOBAL_TRANS)
print("  RES_SCALE        =", cfg.RES_SCALE)
print("  TENDENCY_SCALE   =", cfg.TENDENCY_SCALE)

model = KleinPyGGraphTendencyModelV7(
    nx=cfg.NX,
    ny=cfg.NY,
    in_ch=cfg.IN_CH,
    node_ch=cfg.NODE_CH,
    hidden=cfg.HIDDEN,
    mp_blocks=cfg.MP_BLOCKS,
    radii=tuple(cfg.EDGE_RADII),
    use_global_trans=cfg.USE_GLOBAL_TRANS,
    global_heads=cfg.GLOBAL_TRANS_HEADS,
    global_blocks=cfg.GLOBAL_TRANS_BLOCKS,
    global_token_h=cfg.GLOBAL_TOKEN_H,
    global_token_w=cfg.GLOBAL_TOKEN_W,
    dropout=cfg.DROPOUT,
).to(device)

print("[Model] parameters =", sum(p.numel() for p in model.parameters() if p.requires_grad))

state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:
    print("[Warning] missing keys:", missing)
if unexpected:
    print("[Warning] unexpected keys:", unexpected)
model.eval()
print("[Loaded checkpoint]")

# ============================================================
# HELPERS
# ============================================================
_step_re = re.compile(r"klein_step_(\d+)\.npz")


def step_from_file(path):
    m = _step_re.search(os.path.basename(path))
    return int(m.group(1)) if m is not None else -1


def load_state(path):
    z = np.load(path)
    eta = z["eta"].astype(np.float32)
    u = z["uc"].astype(np.float32)
    v = z["vc"].astype(np.float32)
    t = float(z["t"]) if "t" in z else float(step_from_file(path) * 30.0)
    return np.stack([eta, u, v], axis=0), t


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def kinetic_energy(state):
    eta, u, v = state[0], state[1], state[2]
    h = np.maximum(cfg.H + eta, 1.0)
    return float(np.mean(0.5 * h * (u * u + v * v)))

def total_energy(state):
    eta = state[0]
    u = state[1]
    v = state[2]
    h = np.maximum(cfg.H + eta, 1.0)

    # Full shallow-water energy:
    # KE = 0.5*h*(u^2+v^2)
    # PE anomaly = 0.5*g*eta^2
    return float(np.mean(
        0.5 * h * (u*u + v*v)
        + 0.5 * 9.81 * eta*eta
    ))


def divergence_np(state):
    u = state[1]
    v = state[2]
    dudx = (np.roll(u, -1, axis=-1) - np.roll(u, 1, axis=-1)) / (2.0 * cfg.DX)
    dvdy = (np.roll(v, -1, axis=-2) - np.roll(v, 1, axis=-2)) / (2.0 * cfg.DY)
    return dudx + dvdy


def div_spec_error_np(pred, truth):
    dp = divergence_np(pred)
    dt = divergence_np(truth)
    fp = np.fft.rfft2(dp)
    ft = np.fft.rfft2(dt)
    denom = np.mean(np.abs(ft) ** 2) + 1.0e-20
    return float(np.mean((np.abs(fp) - np.abs(ft)) ** 2) / denom)


def eta_tendency_rmse_np(pred_tend, truth_next, x_prev):
    eta_t_true = (truth_next[0] - x_prev[0]) / cfg.DT_MACRO
    return rmse(pred_tend[0], eta_t_true)

def vmean_metric_np(state):
    return float(np.mean(state[2]))


def eta_smooth_metric_np(state):
    eta = state[0]
    lap = (
        -4.0 * eta
        + np.roll(eta, 1, axis=-1)
        + np.roll(eta, -1, axis=-1)
        + np.roll(eta, 1, axis=-2)
        + np.roll(eta, -1, axis=-2)
    )
    return float(np.mean(lap * lap))


def ke_simple_np(state):
    u = state[1]
    v = state[2]
    return float(np.mean(0.5 * (u*u + v*v)))

def hemi_eta_metric_np(state):
    eta = state[0]
    ny = eta.shape[-2]

    south = eta[:ny//2, :]
    north = eta[ny//2:, :]

    return float(np.mean(north) - np.mean(south))


def y_center_metric_np(state):
    v = state[2]
    ny = v.shape[0]
    y = np.linspace(-1.0, 1.0, ny)[:, None]

    # same mask idea as training
    mask = 1.0 - np.exp(-(y / 0.35) ** 2)

    return float(np.mean(mask * v * v))


def apply_eval_clamp(x):
    if not cfg.USE_EVAL_CLAMP:
        return x
    x_new = x.clone()
    x_new[:, 0:1] = torch.clamp(x_new[:, 0:1], -cfg.ETA_EVAL_CLAMP, cfg.ETA_EVAL_CLAMP)
    x_new[:, 1:2] = torch.clamp(x_new[:, 1:2], -cfg.UV_EVAL_CLAMP, cfg.UV_EVAL_CLAMP)
    x_new[:, 2:3] = torch.clamp(x_new[:, 2:3], -cfg.UV_EVAL_CLAMP, cfg.UV_EVAL_CLAMP)
    return x_new


def laplacian_periodic_x_reflect_y(q):
    q_xp = torch.roll(q, shifts=-1, dims=-1)
    q_xm = torch.roll(q, shifts=1, dims=-1)
    q_yp = torch.empty_like(q)
    q_ym = torch.empty_like(q)
    q_yp[:, :, :-1, :] = q[:, :, 1:, :]
    q_yp[:, :, -1, :] = q[:, :, -1, :]
    q_ym[:, :, 1:, :] = q[:, :, :-1, :]
    q_ym[:, :, 0, :] = q[:, :, 0, :]
    return ((q_xp - 2.0 * q + q_xm) / cfg.DX**2 +
            (q_yp - 2.0 * q + q_ym) / cfg.DY**2)


def apply_inference_diffusion(x):
    if not cfg.USE_INFERENCE_DIFFUSION:
        return x
    x_new = x.clone()
    if cfg.DIFFUSE_ETA:
        eta = x[:, 0:1]
        x_new[:, 0:1] = eta + cfg.NU_INFER * cfg.DT_MACRO * laplacian_periodic_x_reflect_y(eta)
    if cfg.DIFFUSE_UV:
        uv = x[:, 1:3]
        x_new[:, 1:3] = uv + cfg.NU_INFER * cfg.DT_MACRO * laplacian_periodic_x_reflect_y(uv)
    return x_new


def find_ic_dirs():
    all_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])
    selected = []
    for key in cfg.HELDOUT_KEYS:
        matches = [d for d in all_dirs if os.path.basename(d) == key]
        if len(matches) == 0:
            print(f"[Warning] held-out IC not found: {key}")
        else:
            selected.append(matches[0])
    return selected

# ============================================================
# PLOTS
# ============================================================
def plot_rmse(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["eta_rmse"], label="eta")
    plt.plot(df_ic["lead_index"], df_ic["u_rmse"], label="u")
    plt.plot(df_ic["lead_index"], df_ic["v_rmse"], label="v")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("RMSE")
    plt.title(f"RMSE rollout: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_rmse.png"), dpi=160, bbox_inches="tight")
    plt.close()


def plot_ke(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["fd_ke"], label="FD KE")
    plt.plot(df_ic["lead_index"], df_ic["ml_ke"], label="ML KE")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("domain-mean KE")
    plt.title(f"KE rollout: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_ke.png"), dpi=160, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["ke_ratio"])
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("ML KE / FD KE")
    plt.title(f"KE ratio: {ic_key}")
    plt.grid(True)
    plt.savefig(os.path.join(outdir, f"{ic_key}_ke_ratio.png"), dpi=160, bbox_inches="tight")
    plt.close()

def plot_total_energy(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["fd_te_norm"], label="FD TE / FD TE0")
    plt.plot(df_ic["lead_index"], df_ic["ml_te_norm"], label="ML TE / ML TE0")
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("normalized total energy")
    plt.title(f"Total energy normalized by initial value: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_total_energy_norm.png"), dpi=160, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["te_ratio"], label="ML TE / FD TE")
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("total energy ratio")
    plt.title(f"Total energy ratio: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_total_energy_ratio.png"), dpi=160, bbox_inches="tight")
    plt.close()

def plot_divergence(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["div_rmse"], label="div RMSE")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("RMSE")
    plt.title(f"Divergence RMSE rollout: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_div_rmse.png"), dpi=160, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["div_spec_error"], label="div spectrum error")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("normalized spectral error")
    plt.title(f"Divergence spectrum error: {ic_key}")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outdir, f"{ic_key}_div_spec_error.png"), dpi=160, bbox_inches="tight")
    plt.close()


def plot_final_fields(truth, pred, ic_key, lead_index, outdir):
    names = ["eta", "u", "v"]
    for c, name in enumerate(names):
        err = pred[c] - truth[c]
        fdmax = np.nanmax(np.abs(truth[c]))
        mlmax = np.nanmax(np.abs(pred[c]))
        evmax = np.nanmax(np.abs(err))
        if fdmax == 0 or not np.isfinite(fdmax):
            fdmax = 1.0
        if mlmax == 0 or not np.isfinite(mlmax):
            mlmax = 1.0
        if evmax == 0 or not np.isfinite(evmax):
            evmax = 1.0

        fig, axs = plt.subplots(1, 3, figsize=(15, 4))
        im0 = axs[0].imshow(truth[c], origin="lower", cmap="RdBu_r", vmin=-fdmax, vmax=fdmax)
        axs[0].set_title(f"FD {name}")
        plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)
        im1 = axs[1].imshow(pred[c], origin="lower", cmap="RdBu_r", vmin=-mlmax, vmax=mlmax)
        axs[1].set_title(f"ML {name}")
        plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)
        im2 = axs[2].imshow(err, origin="lower", cmap="RdBu_r", vmin=-evmax, vmax=evmax)
        axs[2].set_title(f"ML - FD {name}")
        plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)
        for ax in axs:
            ax.set_xticks([])
            ax.set_yticks([])
        fig.suptitle(f"{ic_key} final lead={lead_index}: {name}")
        plt.savefig(os.path.join(outdir, f"{ic_key}_final_{name}.png"), dpi=170, bbox_inches="tight")
        plt.close()


def save_snapshot_fields(truth, pred, ic_key, lead, outdir):
    fields = [("eta", truth[0], pred[0]), ("u", truth[1], pred[1]), ("v", truth[2], pred[2])]
    for vname, fd_field, ml_field in fields:
        err = ml_field - fd_field
        fdmax = np.nanmax(np.abs(fd_field))
        mlmax = np.nanmax(np.abs(ml_field))
        evmax = np.nanmax(np.abs(err))
        if fdmax == 0 or not np.isfinite(fdmax):
            fdmax = 1.0
        if mlmax == 0 or not np.isfinite(mlmax):
            mlmax = 1.0
        if evmax == 0 or not np.isfinite(evmax):
            evmax = 1.0

        fig, axs = plt.subplots(1, 3, figsize=(18, 5))
        im0 = axs[0].imshow(fd_field, origin="lower", cmap="RdBu_r", vmin=-fdmax, vmax=fdmax)
        axs[0].set_title(f"FD {vname}")
        im1 = axs[1].imshow(ml_field, origin="lower", cmap="RdBu_r", vmin=-mlmax, vmax=mlmax)
        axs[1].set_title(f"ML {vname}")
        im2 = axs[2].imshow(err, origin="lower", cmap="RdBu_r", vmin=-evmax, vmax=evmax)
        axs[2].set_title(f"ML - FD {vname}")
        plt.colorbar(im0, ax=axs[0], fraction=0.046)
        plt.colorbar(im1, ax=axs[1], fraction=0.046)
        plt.colorbar(im2, ax=axs[2], fraction=0.046)
        plt.suptitle(f"{ic_key} lead={lead}: {vname}")
        plt.savefig(os.path.join(outdir, f"{ic_key}_lead{lead:02d}_{vname}.png"), dpi=150, bbox_inches="tight")
        plt.close()

# ============================================================
# MAIN EVALUATION
# ============================================================
ic_dirs = find_ic_dirs()
print("[ICs selected]")
for d in ic_dirs:
    print(" ", os.path.basename(d))

all_rows = []
summary_rows = []

with torch.no_grad():
    for ic_dir in ic_dirs:
        ic_key = os.path.basename(ic_dir)
        print("\nEvaluating:", ic_key)
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")), key=step_from_file)
        if len(files) < 2:
            print("  Skipping: not enough snapshots")
            continue
        if cfg.MAX_ROLLOUT_STEPS is not None:
            files = files[:cfg.MAX_ROLLOUT_STEPS + 1]

        x0_np, t0 = load_state(files[0])
        x = torch.from_numpy(x0_np[None]).to(device)

        fd_te0 = total_energy(x0_np)
        ml_te0 = total_energy(x0_np)

        rows = [{
            "ic_key": ic_key,
            "lead_index": 0,
            "fd_step": step_from_file(files[0]),
            "time_sec": t0,
            "eta_rmse": 0.0,
            "u_rmse": 0.0,
            "v_rmse": 0.0,
            "fd_ke": kinetic_energy(x0_np),
            "ml_ke": kinetic_energy(x0_np),
            "ke_ratio": 1.0,
            "div_rmse": 0.0,
            "div_spec_error": 0.0,
            "eta_tend_rmse": 0.0,
        }]

        final_truth = x0_np
        final_pred = x0_np.copy()

        for lead in range(1, len(files)):
            prev_pred = x.detach().cpu().numpy()[0]
            x, tend = model.forward_one_step(x, cfg.DT_MACRO)
            x = apply_eval_clamp(x)
            x = apply_inference_diffusion(x)

            pred = x.detach().cpu().numpy()[0]
            pred_tend = tend.detach().cpu().numpy()[0]
            truth, t = load_state(files[lead])

            fd_ke = kinetic_energy(truth)
            ml_ke = kinetic_energy(pred)
            ke_ratio = ml_ke / fd_ke if fd_ke != 0 else np.nan

            fd_te = total_energy(truth)
            ml_te = total_energy(pred)
            fd_te_norm = fd_te / fd_te0 if fd_te0 != 0 else np.nan
            ml_te_norm = ml_te / ml_te0 if ml_te0 != 0 else np.nan
            te_ratio = ml_te / fd_te if fd_te != 0 else np.nan

            div_rmse = rmse(divergence_np(pred), divergence_np(truth))
            div_spec_error = div_spec_error_np(pred, truth)
            eta_tend_rmse = eta_tendency_rmse_np(pred_tend, truth, prev_pred)
            vmean_ml = vmean_metric_np(pred)
            vmean_fd = vmean_metric_np(truth)

            eta_smooth_ml = eta_smooth_metric_np(pred)
            eta_smooth_fd = eta_smooth_metric_np(truth)

            ke_simple_ml = ke_simple_np(pred)
            ke_simple_fd = ke_simple_np(truth)
            ke_simple_ratio = ke_simple_ml / (ke_simple_fd + 1e-12)

            ycenter_ml = y_center_metric_np(pred)
            ycenter_fd = y_center_metric_np(truth)
            ycenter_ratio = ycenter_ml / (ycenter_fd + 1e-12)

            hemi_eta_ml = hemi_eta_metric_np(pred)
            hemi_eta_fd = hemi_eta_metric_np(truth)
            hemi_eta_error = hemi_eta_ml - hemi_eta_fd

            row = {
                "ic_key": ic_key,
                "lead_index": lead,
                "fd_step": step_from_file(files[lead]),
                "time_sec": t,
                "eta_rmse": rmse(pred[0], truth[0]),
                "u_rmse": rmse(pred[1], truth[1]),
                "v_rmse": rmse(pred[2], truth[2]),
                "fd_ke": fd_ke,
                "ml_ke": ml_ke,
                "ke_ratio": ke_ratio,
                "div_rmse": div_rmse,
                "div_spec_error": div_spec_error,
                "eta_tend_rmse": eta_tend_rmse,
                "ycenter_ml": ycenter_ml,
                "ycenter_fd": ycenter_fd,
                "ycenter_ratio": ycenter_ratio,
                "vmean_ml": vmean_ml,
                "vmean_fd": vmean_fd,
                "eta_smooth_ml": eta_smooth_ml,
                "eta_smooth_fd": eta_smooth_fd,
                "ke_simple_ratio": ke_simple_ratio,
                "hemi_eta_ml": hemi_eta_ml,
                "hemi_eta_fd": hemi_eta_fd,
                "hemi_eta_error": hemi_eta_ml - hemi_eta_fd,
                "fd_te": fd_te0,
                "ml_te": ml_te0,
                "fd_te_norm": 1.0,
                "ml_te_norm": 1.0,
                "te_ratio": 1.0,
                "fd_te": fd_te,
                "ml_te": ml_te,
                "fd_te_norm": fd_te_norm,
                "ml_te_norm": ml_te_norm,
                "te_ratio": te_ratio,
            }
            rows.append(row)

            print(
                f"[{ic_key}] lead={lead:02d} "
                f"eta={row['eta_rmse']:.3e} "
                f"u={row['u_rmse']:.3e} "
                f"v={row['v_rmse']:.3e} "
                f"KEr={row['ke_ratio']:.3f} "
                f"div={row['div_rmse']:.3e} "
                f"eta_t={row['eta_tend_rmse']:.3e}"
                f"yc={row['ycenter_ratio']:.3f}"
                f"vmean={row['vmean_ml']:+.2e} "
                f"kes={row['ke_simple_ratio']:.3f} "
                f"hemi={row['hemi_eta_error']:+.2e} "
                f"TEr={row['te_ratio']:.3f} "
            )

            if lead in cfg.SAVE_LEADS:
                save_snapshot_fields(truth, pred, ic_key, lead, cfg.SNAPSHOT_DIR)

            final_truth = truth
            final_pred = pred

        df_ic = pd.DataFrame(rows)
        all_rows.extend(rows)
        ic_outdir = os.path.join(cfg.OUT_DIR, ic_key)
        os.makedirs(ic_outdir, exist_ok=True)
        csv_path = os.path.join(ic_outdir, f"{ic_key}_rollout_metrics.csv")
        df_ic.to_csv(csv_path, index=False)

        plot_rmse(df_ic, ic_key, ic_outdir)
        plot_ke(df_ic, ic_key, ic_outdir)
        plot_total_energy(df_ic, ic_key, ic_outdir)
        plot_divergence(df_ic, ic_key, ic_outdir)
        plot_final_fields(final_truth, final_pred, ic_key, len(files) - 1, ic_outdir)

        last = df_ic.iloc[-1]
        summary_rows.append({
            "ic_key": ic_key,
            "n_leads": len(files) - 1,
            "final_step": int(last["fd_step"]),
            "final_eta_rmse": float(last["eta_rmse"]),
            "final_u_rmse": float(last["u_rmse"]),
            "final_v_rmse": float(last["v_rmse"]),
            "final_fd_ke": float(last["fd_ke"]),
            "final_ml_ke": float(last["ml_ke"]),
            "final_ke_ratio": float(last["ke_ratio"]),
            "final_div_rmse": float(last["div_rmse"]),
            "final_div_spec_error": float(last["div_spec_error"]),
            "final_eta_tend_rmse": float(last["eta_tend_rmse"]),
            "final_ycenter_ml": float(last["ycenter_ml"]),
            "final_ycenter_fd": float(last["ycenter_fd"]),
            "final_ycenter_ratio": float(last["ycenter_ratio"]),
            "final_vmean_ml": float(last["vmean_ml"]),
            "final_vmean_fd": float(last["vmean_fd"]),
            "final_eta_smooth_ml": float(last["eta_smooth_ml"]),
            "final_eta_smooth_fd": float(last["eta_smooth_fd"]),
            "final_ke_simple_ratio": float(last["ke_simple_ratio"]),
            "final_hemi_eta_ml": float(last["hemi_eta_ml"]),
            "final_hemi_eta_fd": float(last["hemi_eta_fd"]),
            "final_hemi_eta_error": float(last["hemi_eta_error"]),
            "final_fd_te": float(last["fd_te"]),
            "final_ml_te": float(last["ml_te"]),
            "final_fd_te_norm": float(last["fd_te_norm"]),
            "final_ml_te_norm": float(last["ml_te_norm"]),
            "final_te_ratio": float(last["te_ratio"]),
            "metrics_csv": csv_path,
        })

# ============================================================
# SAVE SUMMARY
# ============================================================
df_all = pd.DataFrame(all_rows)
df_summary = pd.DataFrame(summary_rows)

all_csv = os.path.join(cfg.OUT_DIR, "all_rollout_metrics.csv")
summary_csv = os.path.join(cfg.OUT_DIR, "summary_eval.csv")

df_all.to_csv(all_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

print("\n[DONE]")
print("All metrics:", all_csv)
print("Summary:", summary_csv)
print("\nSummary table:")
try:
    display(df_summary)
except Exception:
    print(df_summary)


torch_geometric version: 2.7.0
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking data roots...
  /content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_adaptive_combined
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_adaptive_combined
     valid IC dirs = 8, total pairs = 192
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
     valid IC dirs = 68, total pairs = 1632
[AutoDetect] using DATA_DIR: /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
[Load] checkpoint: /content/drive/MyDrive/AI_GRAPH_TRANS2/training_runs/klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean/last_model.pt
[Load] updating config from checkpoint
[Config]
  RUN_NAME         = klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean
  CKPT_PATH        = /content/drive/MyDrive/AI_GRAPH_TRANS2/training_runs/klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean/last_model.pt
  NX, NY           = 256 128
  NODE_CH          = 96
  HIDDEN     

/tmp/ipykernel_915/1133795608.py:263: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=blocks)


[Model] parameters = 934435
[Warning] unexpected keys: ['query_mlp.0.weight', 'query_mlp.0.bias', 'query_mlp.2.weight', 'query_mlp.2.bias', 'query_mlp.4.weight', 'query_mlp.4.bias']
[Loaded checkpoint]
[ICs selected]
  test_rh_00
  wave_00
  wave_01
  mix_04

Evaluating: test_rh_00
[test_rh_00] lead=01 eta=2.951e+01 u=1.141e+00 v=1.872e+00 KEr=0.947 div=2.257e-05 eta_t=3.596e-01yc=0.956vmean=-6.56e-02 kes=0.957 hemi=+1.83e-02 TEr=1.020 
[test_rh_00] lead=02 eta=3.761e+01 u=2.125e+00 v=3.997e+00 KEr=0.923 div=4.524e-05 eta_t=3.441e-01yc=0.914vmean=-1.79e-02 kes=0.935 hemi=+3.68e-01 TEr=1.016 
[test_rh_00] lead=03 eta=6.784e+01 u=3.314e+00 v=5.036e+00 KEr=0.915 div=4.635e-05 eta_t=3.917e-01yc=0.878vmean=-6.74e-02 kes=0.928 hemi=+4.06e+00 TEr=1.015 
[test_rh_00] lead=04 eta=9.030e+01 u=4.598e+00 v=5.884e+00 KEr=0.884 div=5.024e-05 eta_t=4.410e-01yc=0.847vmean=-1.54e-01 kes=0.906 hemi=+8.06e+00 TEr=1.060 
[test_rh_00] lead=05 eta=1.149e+02 u=5.864e+00 v=6.439e+00 KEr=0.822 div=4.969e-05 et

,ic_key,n_leads,final_step,final_eta_rmse,final_u_rmse,final_v_rmse,final_fd_ke,final_ml_ke,final_ke_ratio,final_div_rmse,...,final_ke_simple_ratio,final_hemi_eta_ml,final_hemi_eta_fd,final_hemi_eta_error,final_fd_te,final_ml_te,final_fd_te_norm,final_ml_te_norm,final_te_ratio,metrics_csv
0,test_rh_00,24,1200,400.289459,37.834595,25.075102,109153.281250,1.116716e+06,10.230713,0.000114,...,8.543336,65.521027,0.000000,65.521027,201837.906250,1.799985e+06,1.217093,10.853999,8.917972,/content/drive/MyDrive/AI_GRAPH_TRANS2/trainin...
1,wave_00,24,1200,185.515060,8.598119,8.587913,29821.347656,9.266312e+04,3.107275,0.000059,...,2.706513,-20.117352,-0.279531,-19.837821,52204.699219,2.544535e+05,1.288334,6.279532,4.874149,/content/drive/MyDrive/AI_GRAPH_TRANS2/trainin...
2,wave_01,24,1200,227.846527,16.898535,8.535343,88951.671875,1.600184e+05,1.798937,0.000066,...,1.549738,-28.713009,5.710740,-34.423749,117148.367188,3.956562e+05,1.126900,3.805985,3.377394,/content/drive/MyDrive/AI_GRAPH_TRANS2/trainin...
3,mix_04,24,1200,21.070557,2.288892,1.745062,3448.384277,3.169730e+03,0.919193,0.000008,...,0.919309,-6.835601,-1.615912,-5.219689,4474.001465,5.467533e+03,1.067281,1.304289,1.222068,/content/drive/MyDrive/AI_GRAPH_TRANS2/trainin...


# New Section

In [10]:
import torch

ckpt_path = "/content/drive/MyDrive/AI_GRAPH_TRANS2/training_runs/klein_pyg_graph_mr1248_mp6_trans_adapt_v16_clean/last_model.pt"

ckpt = torch.load(ckpt_path, map_location="cpu")

state = ckpt["model_state_dict"]

print("\nTransformer-related keys:\n")

for k in sorted(state.keys()):
    if "global_trans" in k:
        print(k, state[k].shape)


Transformer-related keys:

global_trans.encoder.layers.0.linear1.bias torch.Size([384])
global_trans.encoder.layers.0.linear1.weight torch.Size([384, 96])
global_trans.encoder.layers.0.linear2.bias torch.Size([96])
global_trans.encoder.layers.0.linear2.weight torch.Size([96, 384])
global_trans.encoder.layers.0.norm1.bias torch.Size([96])
global_trans.encoder.layers.0.norm1.weight torch.Size([96])
global_trans.encoder.layers.0.norm2.bias torch.Size([96])
global_trans.encoder.layers.0.norm2.weight torch.Size([96])
global_trans.encoder.layers.0.self_attn.in_proj_bias torch.Size([288])
global_trans.encoder.layers.0.self_attn.in_proj_weight torch.Size([288, 96])
global_trans.encoder.layers.0.self_attn.out_proj.bias torch.Size([96])
global_trans.encoder.layers.0.self_attn.out_proj.weight torch.Size([96, 96])
global_trans.encoder.layers.1.linear1.bias torch.Size([384])
global_trans.encoder.layers.1.linear1.weight torch.Size([384, 96])
global_trans.encoder.layers.1.linear2.bias torch.Size([96